# 响应后的后台任务

学习目标：注册响应后的有限任务，区分任务执行位置、完成结果和失败，并让任务自行管理文件资源。

前置知识：函数、文件读写、异常处理、协程、依赖注入与应用生命周期。

运行环境：Python 3.12、FastAPI 0.141.1、Starlette 1.6.0。

工作目录：`content/Web与应用开发/FastAPI`；前四节使用应用内调用，第五节启动本地 8180 端口。输入都是本章提供的小文本，不发送邮件或调用外部服务。

环境准备：[FastAPI 环境与运行说明](README.md)。

配套脚本：位于 scripts/18-background-tasks/。

1. [app.py](scripts/18-background-tasks/app.py)：本章末尾已经讲解的文件任务与控制接口，用于观察真实响应先于任务完成。

## 1 注册任务时传入函数

BackgroundTasks 用来给响应附加任务。FastAPI 按参数类型提供这个对象；add_task 接收函数及其参数，先登记工作，响应发送后再执行。传入 save_notice，不写成 save_notice(...)，否则会当场调用函数。

先把一条短消息加入内存列表，观察任务是否完成。202 表示请求已接受处理，不承诺处理已经成功。

In [1]:
from fastapi import BackgroundTasks, FastAPI
from fastapi.testclient import TestClient

app = FastAPI()
notices = []


def save_notice(message: str):
    notices.append(message)


@app.post('/notice', status_code=202)
async def queue_notice(tasks: BackgroundTasks):
    tasks.add_task(save_notice, '学习记录已提交')
    return {'status': 'accepted'}


with TestClient(app) as client:
    response = client.post('/notice')
assert response.status_code == 202
assert notices == ['学习记录已提交']
print(response.json(), notices)  # 预期：{'status': 'accepted'} ['学习记录已提交']。

{'status': 'accepted'} ['学习记录已提交']


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


这里 TestClient 的调用返回时，已经等到应用调用结束，所以列表已有结果。这可以检查副作用，不能用它证明真实客户端必须等待任务。第五节再用真实端口分开观察响应和任务完成。

## 2 同步函数、异步函数和执行顺序

同步任务通过线程池执行；异步任务在应用的事件循环中被 await。异步函数里直接写阻塞 I/O，并不会因为被称为“后台任务”就自动转到线程。后台任务仍在同一个服务进程内，也不等于创建计算进程池。

下面分别记录线程标识，不把具体标识数值作为预期输出。

In [2]:
import threading

positions = {}
order = []


def sync_work():
    positions['sync'] = threading.get_ident()
    order.append('sync')


async def async_work():
    positions['async'] = threading.get_ident()
    order.append('async')


@app.post('/positions')
async def queue_positions(tasks: BackgroundTasks):
    positions['route'] = threading.get_ident()
    tasks.add_task(sync_work)
    tasks.add_task(async_work)
    return {'registered': 2}

同一响应上的多个任务按添加顺序执行，不会自动并行。顺序只描述这一响应内的任务，不表示不同请求之间存在全局顺序。

In [3]:
with TestClient(app) as client:
    response = client.post('/positions')
assert response.status_code == 200
assert order == ['sync', 'async']
assert positions['route'] == positions['async']
assert positions['route'] != positions['sync']
print('任务顺序：', order)  # 预期：任务顺序： ['sync', 'async']。
print('异步任务与路由同一线程：', positions['route'] == positions['async'])  # 预期：异步任务与路由同一线程： True。
print('同步任务与路由不同线程：', positions['route'] != positions['sync'])  # 预期：同步任务与路由不同线程： True。

任务顺序： ['sync', 'async']
异步任务与路由同一线程： True
同步任务与路由不同线程： True


## 3 依赖可以登记任务，任务自己打开资源

依赖和路由可以使用同一请求的 BackgroundTasks，登记的任务会合并。本例让依赖登记一条审计消息，路由登记文件工作。

文件任务接收路径与文本，在自身的 with 块中打开并关闭文件。不要把请求依赖借出的文件句柄或数据库会话交给后台任务；需要数据库时，可以传记录编号，让任务使用自己的会话查询。

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Annotated

from fastapi import Depends

audit = []


def record_request(tasks: BackgroundTasks):
    tasks.add_task(audit.append, '请求已接受')


def write_note(path: Path, text: str):
    with path.open('w', encoding='utf-8') as stream:
        stream.write(text)


@app.post('/file', status_code=202)
async def queue_file(tasks: BackgroundTasks, _: Annotated[None, Depends(record_request)]):
    tasks.add_task(write_note, note_path, '一条学习记录')
    return {'status': 'accepted'}

note_path 是本次测试准备的临时文件路径，不来自客户端输入。临时目录覆盖请求与后台任务的整个执行过程，离开 with 后统一删除。

In [5]:
with TemporaryDirectory(prefix='fastapi18-note-') as folder:
    note_path = Path(folder) / 'note.txt'
    with TestClient(app) as client:
        response = client.post('/file')
    assert response.status_code == 202
    assert note_path.read_text(encoding='utf-8') == '一条学习记录'
    assert audit == ['请求已接受']
    print(audit, note_path.read_text(encoding='utf-8'))  # 预期：['请求已接受'] 一条学习记录。
assert not Path(folder).exists()
print('临时文件已清理')  # 预期：临时文件已清理。

['请求已接受'] 一条学习记录
临时文件已清理


## 4 任务失败不能修改已经发送的响应

同一响应上的后台任务按顺序执行；一个任务抛出未处理异常，其后任务不再运行。此时响应已经发送，不能将客户端收到的 202 改成 500，因此 accepted 与工作成功完成必须分开判断。

![响应已发送，后台任务才继续](image/illustration/18-01-background-failure.svg)

图示：本例后台失败路径。最后一个框表示被跳过的任务，不表示它会继续执行。

下面故意让首个任务失败，用 failure\_order 核对后续任务没有执行。raise\_server\_exceptions=False 只为观察已经发送的响应；默认 TestClient 会重新抛出应用异常。真实端口实验再区分接收与完成两个时刻。

In [6]:
failure_order = []


def fail_work():
    failure_order.append('失败任务')
    raise RuntimeError('本章故意触发的失败')


@app.post('/failure', status_code=202)
async def queue_failure(tasks: BackgroundTasks):
    tasks.add_task(fail_work)
    tasks.add_task(failure_order.append, '后续任务')
    return {'status': 'accepted'}


with TestClient(app, raise_server_exceptions=False) as client:
    response = client.post('/failure')
assert response.status_code == 202
assert failure_order == ['失败任务']
print(response.status_code, failure_order)  # 预期：202 ['失败任务']。

202 ['失败任务']


对于任务预计可能发生的文件错误，可以在任务里捕获 OSError 并明确记录失败；不要只吞掉异常却仍标记成功。本例用一个结果列表观察失败，不实现重试或任务查询系统。

In [7]:
outcomes = []


def guarded_write(path: Path, text: str):
    try:
        write_note(path, text)
    except OSError:
        outcomes.append('写入失败')
    else:
        outcomes.append('写入成功')


with TemporaryDirectory(prefix='fastapi18-failure-') as folder:
    tasks = BackgroundTasks()
    # 把目录当文件写入会触发 OSError；不同系统的子异常名称可能不同。
    tasks.add_task(guarded_write, Path(folder), '不会写入')
    tasks.add_task(outcomes.append, '已记录结果')
    await tasks()  # 这里直接运行任务集合，观察函数行为，不发送 HTTP。
assert outcomes == ['写入失败', '已记录结果']
assert not Path(folder).exists()
print(outcomes)  # 预期：['写入失败', '已记录结果']。

['写入失败', '已记录结果']


## 5 用真实端口分开观察接受与完成

用 threading.Event 作为一次实验的“继续”信号：clear 清除信号，set 发出信号，wait(10) 最多等待 10 秒。这样可以先收到响应，再允许写文件，不用比较机器上的固定耗时。

先定义新的小应用。lifespan 在服务启动时创建临时目录、信号和状态，退出时删除目录。state 只保存本进程的演示状态，当前例子只接收一个待完成任务；不启用多 worker。

In [8]:
from contextlib import asynccontextmanager
import logging

from fastapi import HTTPException
from starlette.concurrency import run_in_threadpool

logger = logging.getLogger('uvicorn.error')


@asynccontextmanager
async def lifespan(app: FastAPI):
    with TemporaryDirectory(prefix='fastapi18-service-') as folder:
        app.state.path = Path(folder) / 'note.txt'
        app.state.gate = threading.Event()
        app.state.status = 'idle'
        yield
    # 预期：正常退出后日志显示教学临时目录已删除：True。
    logger.info('教学临时目录已删除：%s', not Path(folder).exists())


app = FastAPI(lifespan=lifespan)

文件任务自己打开资源。等待超时和文件错误会把状态改为 failed；只有 with 正常退出、文件关闭之后才标记 done。fail 参数仅用于本地失败实验，为 true 时故意把目录作为目标。

In [9]:
def finish_file(fail: bool = False):
    try:
        if not app.state.gate.wait(10):
            raise TimeoutError('未收到继续信号')
        path = app.state.path.parent if fail else app.state.path
        with path.open('w', encoding='utf-8') as stream:
            stream.write('后台文件工作完成')
    except (OSError, TimeoutError):
        app.state.status = 'failed'
        # 预期：故意触发文件错误或等待超时时显示警告，任务状态为 failed。
        logger.warning('文件任务失败')
    else:
        app.state.status = 'done'

后台入口只登记任务并返回 accepted。等待状态下再次提交返回 409；本例的状态和信号仅供单用户顺序实验，不提供持久化任务队列的并发保证。

In [10]:
@app.post('/jobs/background', status_code=202)
async def background_job(tasks: BackgroundTasks, fail: bool = False):
    if app.state.status == 'waiting':
        raise HTTPException(409, '已有待完成任务')
    app.state.gate.clear()
    app.state.status = 'waiting'
    tasks.add_task(finish_file, fail)
    return {'status': 'accepted'}


@app.post('/release')
async def release_job():
    app.state.gate.set()
    return {'released': True}

对照入口直接 await 文件工作，返回时工作已经结束。run_in_threadpool 把同步等待和文件操作交给线程池，await 仍然等待其完成；线程切换不会把它变成“响应后才做”。

状态查询使用同步路由读取小文件；仅在 done 时读取文本，避免把上次文件内容当作本次成功结果。

In [11]:
@app.post('/jobs/await')
async def awaited_job():
    if app.state.status == 'waiting':
        raise HTTPException(409, '已有待完成任务')
    app.state.status = 'waiting'
    app.state.gate.set()
    await run_in_threadpool(finish_file)
    return {'status': app.state.status}


@app.get('/state')
def read_state():
    text = None
    if app.state.status == 'done':
        text = app.state.path.read_text(encoding='utf-8')
    return {'status': app.state.status, 'text': text}

以上应用定义也整理在配套 app.py 中，独立服务与 Notebook 内核各自拥有自己的应用实例。

Step 1：在课程目录的独立终端启动服务，保留这个终端。

```powershell
python -m uvicorn app:app --app-dir scripts/18-background-tasks --host 127.0.0.1 --port 8180
```

Step 2：等服务显示启动完成，回到 Notebook 执行下面的真实 HTTP 调用。

先准备一个有截止时间的状态查询函数。max_wait 表示本次轮询最多等待的秒数；HTTPX 的 timeout 单独限制网络操作等待。循环用于等待明确的 done 或 failed，不用固定秒数假定任务已完成。

In [12]:
import time
import httpx

BASE_URL = 'http://127.0.0.1:8180'


def wait_result(client: httpx.Client, max_wait: float = 5) -> dict:
    deadline = time.monotonic() + max_wait
    while time.monotonic() < deadline:
        response = client.get('/state')
        response.raise_for_status()
        state = response.json()
        if state['status'] in {'done', 'failed'}:
            return state
        time.sleep(0.05)
    raise TimeoutError('任务未在观察期限内结束')

这一个单元连续完成提交、观察和放行，避免在手工操作时错过 10 秒等待期限。finally 确保检查失败时也发出继续信号；任务自身的 10 秒上限防止永久等待。

In [13]:
with httpx.Client(base_url=BASE_URL, timeout=2, trust_env=False) as client:
    try:
        response = client.post('/jobs/background')
        response.raise_for_status()
        waiting = client.get('/state').json()
        assert response.status_code == 202
        assert waiting == {'status': 'waiting', 'text': None}
        print('响应已收到：', response.status_code, response.json())  # 预期：202 {'status': 'accepted'}。
        print('放行之前：', waiting)  # 预期：{'status': 'waiting', 'text': None}。
    finally:
        client.post('/release').raise_for_status()
    completed = wait_result(client)
    assert completed == {'status': 'done', 'text': '后台文件工作完成'}
    print('放行之后：', completed)  # 预期：{'status': 'done', 'text': '后台文件工作完成'}。

响应已收到： 202 {'status': 'accepted'}
放行之前： {'status': 'waiting', 'text': None}
放行之后： {'status': 'done', 'text': '后台文件工作完成'}


再分别检查任务失败与 await 对照入口。后台失败通过状态查询得知，已经返回的 202 保持不变；await 入口返回 done 时，文件已完成并关闭。

In [14]:
with httpx.Client(base_url=BASE_URL, timeout=2, trust_env=False) as client:
    try:
        response = client.post('/jobs/background', params={'fail': True})
        response.raise_for_status()
    finally:
        client.post('/release').raise_for_status()
    failed = wait_result(client)
    assert response.status_code == 202 and failed['status'] == 'failed'
    print('已返回响应：', response.status_code, '后续结果：', failed)  # 预期：已返回响应： 202 后续结果： {'status': 'failed', 'text': None}。
    awaited = client.post('/jobs/await')
    assert awaited.status_code == 200 and awaited.json() == {'status': 'done'}
    assert client.get('/state').json()['text'] == '后台文件工作完成'
    print('await 入口返回时：', awaited.json())  # 预期：await 入口返回时： {'status': 'done'}。

已返回响应： 202 后续结果： {'status': 'failed', 'text': None}
await 入口返回时： {'status': 'done'}


Step 3：在服务终端按 Ctrl+C，等待应用关闭和临时目录删除的日志；再运行下面单元确认端口已关闭。

In [15]:
try:
    with httpx.Client(timeout=5, trust_env=False) as client:
        client.get(BASE_URL + '/state')
except httpx.ConnectError:
    print('服务端口已关闭')  # 预期：确认无法连接后显示服务端口已关闭。
else:
    raise AssertionError('请先在服务终端关闭本章服务')

服务端口已关闭


## 6 选择后台任务的边界

这个例子适合有限、短小的进程内工作。内存状态没有持久化，进程退出或重启后不能据此恢复待办工作；BackgroundTasks 本身没有替我们提供持久化、自动重试或跨机器调度。

Uvicorn 正常关闭时会等待后台工作结束，但仍受关闭超时限制。进程被强制终止、机器故障或任务超过期限，都不能从“返回了 202”推出任务一定完成。需要可靠保存任务、重试和跨进程处理时，应采用具有相应机制的任务系统，明确任务保存和恢复的方式。

不要把占用大量 CPU 的计算直接搬进异步后台函数。同步后台工作也会占用共享线程容量；“响应后执行”改变的是响应与工作完成的关系，并不会消除资源成本。

## 本章小结

（1）add_task 登记函数和参数；同一响应上的任务按顺序执行，未处理异常会阻止后续任务。

（2）同步任务进入线程池，异步任务在事件循环执行，二者仍属于服务进程。

（3）任务自己创建和关闭文件或数据库资源；预计的失败应明确记录，不能修改已经发送的响应。

（4）应用内测试检查副作用，真实端口实验区分 accepted 与 done；需要可靠重试时另选具备相应能力的任务系统。

自查：收到 202 后，还需要观察什么才能判断任务成功？首个任务失败为何会影响同一响应的后续任务？

## 练习

1. 在第 2 节增加第三个小任务，把顺序标记加入 order。验证顺序与登记顺序一致，并说明它是否使三个任务并行。

In [ ]:
# 在此完成本题；真实服务题结束后关闭服务并核对资源清理。

2. 把第 4 节的 guarded_write 目标改为临时目录下的普通文件。验证结果为“写入成功”，文件内容正确，离开临时目录后资源被清理。

In [ ]:
# 在此完成本题；真实服务题结束后关闭服务并核对资源清理。

3. 启动配套服务，提交后台工作后暂不调用 /release。等待状态从 waiting 变为 failed，确认最初响应仍是 202；完成后关闭服务。可将轮询的 max_wait 设为 12 秒覆盖本例 10 秒期限。

In [ ]:
# 在此完成本题；真实服务题结束后关闭服务并核对资源清理。

4. 在配套后台入口增加另一项“写完以后追加结束标记”的任务，分别模拟第一个任务捕获和不捕获文件错误。检查后续任务是否执行，且不能只凭 202 判断结果。

In [ ]:
# 在此完成本题；真实服务题结束后关闭服务并核对资源清理。

### 第 4 题提示与解析

提示 1：让第一个任务把目录当文件写入，第二个任务只追加一个结束标记；先预测两种异常处理方式的差别。

提示 2：捕获版本用 `guarded_write` 的结构记录失败；不捕获版本让 `OSError` 离开任务。分别核对任务结果和后续标记。

解析：第一个任务内部捕获并记录文件错误后正常返回，任务集合会继续调用下一任务，因此有“写入失败”和结束标记；若异常向外传播，顺序执行被中断，后续标记不会出现。HTTP 响应已经发送，两种后台结果都不能仅凭 202 区分。

第 3 题轮询期限设为 12 秒覆盖 10 秒任务等待；它与单次请求的读取超时不同。修改服务后重启，练习结束恢复示例。

## 参考与引用来源

1. **FastAPI 官方文档**：[Background Tasks](https://fastapi.tiangolo.com/tutorial/background-tasks/) 的 Using BackgroundTasks、Create a task function、Add the background task、Dependency Injection 与 Caveat，支持登记方式、依赖合并和小型后台工作的范围；[Advanced Dependencies](https://fastapi.tiangolo.com/advanced/advanced-dependencies/#background-tasks-and-dependencies-with-yield-technical-details) 中 Background Tasks 的 Tip，支持任务独立创建资源和传递记录编号；[Lifespan Events](https://fastapi.tiangolo.com/advanced/events/#lifespan)，支持启动与关闭时的资源管理。
2. **Starlette 官方文档**：[Background](https://starlette.dev/background/) 的 BackgroundTasks 与 Important，支持进程内、响应后、顺序和失败停止；[Thread Pool](https://starlette.dev/threadpool/)，支持同步后台任务及共享线程容量；[Exceptions](https://starlette.dev/exceptions/#errors-and-handled-exceptions)，支持已发送响应不能由后台异常替换；[TestClient](https://starlette.dev/testclient/)，支持 raise_server_exceptions 和上下文管理。
3. **GitHub 上的 Starlette 官方源码（1.6.0）**：[background.py](https://github.com/Kludex/starlette/blob/1.6.0/starlette/background.py) 的 BackgroundTask.__call__、BackgroundTasks.__call__，支持同步／异步调用及顺序；[concurrency.py](https://github.com/Kludex/starlette/blob/1.6.0/starlette/concurrency.py) 的 run_in_threadpool，支持把同步函数交给线程池并等待结果；[responses.py](https://github.com/Kludex/starlette/blob/1.6.0/starlette/responses.py) 的 Response.__call__ 和 [testclient.py](https://github.com/Kludex/starlette/blob/1.6.0/starlette/testclient.py) 的 _TestClientTransport.handle_request，支持响应消息发送后继续等待后台工作、应用内调用与网络观察的区别。
4. **Python 3.12 官方文档**：[Event Objects](https://docs.python.org/3.12/library/threading.html#event-objects)，支持 set、clear、wait 与秒数期限；[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)，支持临时目录清理；[Reading and Writing Files](https://docs.python.org/3.12/tutorial/inputoutput.html#reading-and-writing-files)，支持 with 关闭文件和显式编码；[time.monotonic](https://docs.python.org/3.12/library/time.html#time.monotonic)，支持轮询截止时间。
5. **HTTPX 官方文档**：[Clients](https://www.python-httpx.org/advanced/clients/) 的 Usage、Base URL，支持客户端上下文与基础地址；[Timeouts](https://www.python-httpx.org/advanced/timeouts/)，支持网络等待与总轮询期限的区分；[Exceptions](https://www.python-httpx.org/exceptions/)，支持 ConnectError 与 raise_for_status。
6. **Uvicorn 官方文档**：[Server Behavior](https://uvicorn.dev/server-behavior/#graceful-process-shutdown) 的 Graceful Process Shutdown，支持正常退出等待任务及超时边界；[Settings](https://uvicorn.dev/settings/)，支持应用入口、app-dir、绑定地址与端口。
7. **RFC Editor**：[RFC 9110 第 15.3.3 节](https://www.rfc-editor.org/rfc/rfc9110.html#section-15.3.3)，支持 202 Accepted 表示接受处理而非处理已完成。